# The Fashion-MNIST Dataset
In this notebook, we make a few attempts at predicting the classes of different fashion items. Here, we use the [Fashion-MNIST dataset](https://github.com/zalandoresearch/fashion-mnist) which consists of images of items belonging to 10 different classes:

| Label | Description |
| --- | --- |
| 0 | T-shirt/top |
| 1 | Trouser |
| 2 | Pullover |
| 3 | Dress |
| 4 | Coat |
| 5 | Sandal |
| 6 | Shirt |
| 7 | Sneaker |
| 8 | Bag |
| 9 | Ankle boot |

Following is what we'll do to predict the classes:
- First, we'll start with a basic feed-forward neural network
- Second, we'll add a few convolution layers along with pooling
- Third, we'll add regularization in the form of batch normalization & dropout
- Fourth, we'll see if data augmentation/weight decay/scheduled learning rate will lead to improvements

## Imports

In [1]:
import numpy as np
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# for Hugging Face CV model
from PIL import Image
from transformers import AutoImageProcessor, AutoModelForImageClassification

In [2]:
# use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


-----
## Helper Functions

In [3]:
def trainer_func(model, criterion, optimizer, train_loader, test_loader, device, epochs=10, scheduler=None):
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)  # needed to use GPU, comment this line if you want to ensure using CPU
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
    
        # Evaluate on test set
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for xb, yb in test_loader:
                outputs = model(xb.to(device))
                _, predicted = torch.max(outputs, 1)
                total += yb.size(0)
                correct += (predicted == yb.to(device)).sum().item()
        acc = correct / total
        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader):.4f}, Test Acc: {acc:.2f}")

        if scheduler:
            scheduler.step()

In [4]:
def tester_func(model, test_loader, device):
    # predict on test set
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(predicted.cpu().numpy())
    
    # evaluate
    classes = [
        "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
        "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
    ]
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=classes))
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))
    
    return

-----
## Load data

In [5]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # normalize to [-1, 1]
])

train_dataset = torchvision.datasets.FashionMNIST(
    root="./data", train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.FashionMNIST(
    root="./data", train=False, download=True, transform=transform
)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64,   shuffle=True)
test_loader  = torch.utils.data.DataLoader(test_dataset,  batch_size=1000, shuffle=False)

-----
## 1. Feed-forward Neural Network

In [6]:
# neural network design
class FashionNet(nn.Module):
    def __init__(self):
        super(FashionNet, self).__init__()
        self.fc1 = nn.Linear(28*28, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.view(-1, 28*28)  # flatten images
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)  # raw logits
        return x

In [7]:
# train model
model = FashionNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
trainer_func(model, criterion, optimizer, train_loader, test_loader, device, epochs=20)

Epoch 1/20, Loss: 0.4932, Test Acc: 0.85
Epoch 2/20, Loss: 0.3669, Test Acc: 0.86
Epoch 3/20, Loss: 0.3271, Test Acc: 0.86
Epoch 4/20, Loss: 0.3046, Test Acc: 0.88
Epoch 5/20, Loss: 0.2864, Test Acc: 0.88
Epoch 6/20, Loss: 0.2706, Test Acc: 0.87
Epoch 7/20, Loss: 0.2555, Test Acc: 0.88
Epoch 8/20, Loss: 0.2419, Test Acc: 0.88
Epoch 9/20, Loss: 0.2312, Test Acc: 0.88
Epoch 10/20, Loss: 0.2225, Test Acc: 0.89
Epoch 11/20, Loss: 0.2093, Test Acc: 0.89
Epoch 12/20, Loss: 0.2018, Test Acc: 0.89
Epoch 13/20, Loss: 0.1924, Test Acc: 0.89
Epoch 14/20, Loss: 0.1852, Test Acc: 0.89
Epoch 15/20, Loss: 0.1778, Test Acc: 0.89
Epoch 16/20, Loss: 0.1708, Test Acc: 0.88
Epoch 17/20, Loss: 0.1607, Test Acc: 0.88
Epoch 18/20, Loss: 0.1576, Test Acc: 0.89
Epoch 19/20, Loss: 0.1497, Test Acc: 0.89
Epoch 20/20, Loss: 0.1476, Test Acc: 0.88


In [8]:
# predict on test set and evaluate
tester_func(model, test_loader, device)

Accuracy: 0.8831

Classification Report:
              precision    recall  f1-score   support

 T-shirt/top       0.77      0.89      0.83      1000
     Trouser       0.99      0.97      0.98      1000
    Pullover       0.81      0.83      0.82      1000
       Dress       0.88      0.91      0.89      1000
        Coat       0.84      0.80      0.82      1000
      Sandal       0.92      0.97      0.94      1000
       Shirt       0.75      0.63      0.68      1000
     Sneaker       0.95      0.93      0.94      1000
         Bag       0.97      0.95      0.96      1000
  Ankle boot       0.97      0.95      0.96      1000

    accuracy                           0.88     10000
   macro avg       0.88      0.88      0.88     10000
weighted avg       0.88      0.88      0.88     10000


Confusion Matrix:
[[886   2  12  25   3  10  54   0   8   0]
 [  5 975   1  13   3   2   0   0   1   0]
 [ 26   1 830  14  53   1  70   0   5   0]
 [ 26   9  11 906  29   2  11   0   6   0]
 [  3   0

We got an overall accuracy of > 85%. 

It seems the model confuses these together:
- shirts & t-shirts
- coats & pullovers
- dresses & shirts/t-shirts

Let's see if we can improve on this.

-----
## 2. Convolution + Pooling

In [9]:
# neural network design
class FashionCNN(nn.Module):
    def __init__(self):
        super(FashionCNN, self).__init__()
        # Conv layer 1: input=1 channel, output=32 filters, kernel=3x3
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        # Conv layer 2: input=32, output=64 filters
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        # Pooling layer (2x2 max pooling)
        self.pool = nn.MaxPool2d(2, 2)
        # Fully connected layers
        self.fc1 = nn.Linear(64 * 7 * 7, 128)  # 28→14→7 after pooling
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        # Conv + ReLU + Pool
        x = F.relu(self.pool(self.conv1(x)))  # [1x28x28] -> [32x14x14]
        x = F.relu(self.pool(self.conv2(x)))  # [32x14x14] -> [64x7x7]
        # Flatten
        x = x.view(-1, 64 * 7 * 7)
        # FC layers
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [10]:
# train model
model = FashionCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
trainer_func(model, criterion, optimizer, train_loader, test_loader, device, epochs=20)

Epoch 1/20, Loss: 0.4421, Test Acc: 0.87
Epoch 2/20, Loss: 0.2813, Test Acc: 0.90
Epoch 3/20, Loss: 0.2347, Test Acc: 0.90
Epoch 4/20, Loss: 0.2046, Test Acc: 0.91
Epoch 5/20, Loss: 0.1796, Test Acc: 0.91
Epoch 6/20, Loss: 0.1545, Test Acc: 0.92
Epoch 7/20, Loss: 0.1380, Test Acc: 0.91
Epoch 8/20, Loss: 0.1164, Test Acc: 0.92
Epoch 9/20, Loss: 0.1018, Test Acc: 0.92
Epoch 10/20, Loss: 0.0866, Test Acc: 0.92
Epoch 11/20, Loss: 0.0742, Test Acc: 0.92
Epoch 12/20, Loss: 0.0602, Test Acc: 0.92
Epoch 13/20, Loss: 0.0531, Test Acc: 0.92
Epoch 14/20, Loss: 0.0440, Test Acc: 0.92
Epoch 15/20, Loss: 0.0393, Test Acc: 0.92
Epoch 16/20, Loss: 0.0365, Test Acc: 0.92
Epoch 17/20, Loss: 0.0317, Test Acc: 0.91
Epoch 18/20, Loss: 0.0284, Test Acc: 0.92
Epoch 19/20, Loss: 0.0240, Test Acc: 0.92
Epoch 20/20, Loss: 0.0261, Test Acc: 0.92


In [11]:
# predict on test set and evaluate
tester_func(model, test_loader, device)

Accuracy: 0.9186

Classification Report:
              precision    recall  f1-score   support

 T-shirt/top       0.87      0.87      0.87      1000
     Trouser       0.99      0.99      0.99      1000
    Pullover       0.87      0.89      0.88      1000
       Dress       0.92      0.91      0.91      1000
        Coat       0.87      0.88      0.88      1000
      Sandal       0.98      0.98      0.98      1000
       Shirt       0.80      0.76      0.78      1000
     Sneaker       0.94      0.98      0.96      1000
         Bag       0.97      0.98      0.98      1000
  Ankle boot       0.98      0.95      0.96      1000

    accuracy                           0.92     10000
   macro avg       0.92      0.92      0.92     10000
weighted avg       0.92      0.92      0.92     10000


Confusion Matrix:
[[873   1  10  10   7   3  84   0  12   0]
 [  1 987   3   5   1   0   2   0   1   0]
 [ 21   1 887   7  39   0  41   0   4   0]
 [ 21   7   8 910  21   0  28   1   4   0]
 [  3   0

We got an overall accuracy of > 90%. 

While accuracy did improve, it seems the model mispredicts a lot of items as shirts AND mispredicts a lot of shirts as something else!

Let's see if we can improve on this further still.

-----
## 3. Dropout + BatchNorm

In [12]:
# neural network design
class FashionCNNImproved(nn.Module):
    def __init__(self):
        super(FashionCNNImproved, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.bn1(self.pool(self.conv1(x))))  # -> [32, 14, 14]
        x = F.relu(self.bn2(self.pool(self.conv2(x))))  # -> [64, 7, 7]
        x = x.view(-1, 64 * 7 * 7)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)  # dropout on FC layer
        x = self.fc2(x)
        return x

In [13]:
# train model
model = FashionCNNImproved().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
trainer_func(model, criterion, optimizer, train_loader, test_loader, device, epochs=20)

Epoch 1/20, Loss: 0.3741, Test Acc: 0.89
Epoch 2/20, Loss: 0.2607, Test Acc: 0.90
Epoch 3/20, Loss: 0.2176, Test Acc: 0.92
Epoch 4/20, Loss: 0.1928, Test Acc: 0.91
Epoch 5/20, Loss: 0.1693, Test Acc: 0.92
Epoch 6/20, Loss: 0.1486, Test Acc: 0.92
Epoch 7/20, Loss: 0.1349, Test Acc: 0.92
Epoch 8/20, Loss: 0.1196, Test Acc: 0.92
Epoch 9/20, Loss: 0.1062, Test Acc: 0.92
Epoch 10/20, Loss: 0.0967, Test Acc: 0.92
Epoch 11/20, Loss: 0.0877, Test Acc: 0.92
Epoch 12/20, Loss: 0.0771, Test Acc: 0.92
Epoch 13/20, Loss: 0.0744, Test Acc: 0.92
Epoch 14/20, Loss: 0.0672, Test Acc: 0.92
Epoch 15/20, Loss: 0.0637, Test Acc: 0.92
Epoch 16/20, Loss: 0.0612, Test Acc: 0.92
Epoch 17/20, Loss: 0.0558, Test Acc: 0.92
Epoch 18/20, Loss: 0.0508, Test Acc: 0.92
Epoch 19/20, Loss: 0.0514, Test Acc: 0.92
Epoch 20/20, Loss: 0.0491, Test Acc: 0.92


In [14]:
# predict on test set and evaluate
tester_func(model, test_loader, device)

Accuracy: 0.9225

Classification Report:
              precision    recall  f1-score   support

 T-shirt/top       0.86      0.89      0.87      1000
     Trouser       0.99      0.98      0.99      1000
    Pullover       0.86      0.88      0.87      1000
       Dress       0.92      0.92      0.92      1000
        Coat       0.89      0.87      0.88      1000
      Sandal       0.99      0.98      0.99      1000
       Shirt       0.79      0.77      0.78      1000
     Sneaker       0.96      0.98      0.97      1000
         Bag       0.98      0.99      0.99      1000
  Ankle boot       0.98      0.97      0.97      1000

    accuracy                           0.92     10000
   macro avg       0.92      0.92      0.92     10000
weighted avg       0.92      0.92      0.92     10000


Confusion Matrix:
[[892   0  17   9   1   1  74   0   6   0]
 [  1 977   1  15   0   0   4   0   2   0]
 [ 19   1 880   7  38   0  55   0   0   0]
 [ 18   3  12 918  24   0  24   0   1   0]
 [  0   0

Well this model did do slightly better than the previous one (in terms of accuracy).

We'll give this one more shot!

-----
## 4. Data Augmentation + Learning Rate Scheduling + Weight Decay

In [15]:
# data augmentation

transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),      # random flip
    transforms.RandomRotation(10),          # random rotation ±10°
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))    # normalize to [-1,1]
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset2 = torchvision.datasets.FashionMNIST(
    root="./data", train=True, download=True, transform=transform_train
)
train_loader2 = torch.utils.data.DataLoader(train_dataset2, batch_size=64, shuffle=True)

In [16]:
# train model
model = FashionCNNImproved().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)  # weight decay
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)  # learning rate scheduling
trainer_func(model, criterion, optimizer, train_loader2, test_loader, device, epochs=20, scheduler=scheduler)

Epoch 1/20, Loss: 0.4554, Test Acc: 0.89
Epoch 2/20, Loss: 0.3306, Test Acc: 0.90
Epoch 3/20, Loss: 0.2961, Test Acc: 0.90
Epoch 4/20, Loss: 0.2795, Test Acc: 0.90
Epoch 5/20, Loss: 0.2644, Test Acc: 0.91
Epoch 6/20, Loss: 0.2538, Test Acc: 0.91
Epoch 7/20, Loss: 0.2468, Test Acc: 0.91
Epoch 8/20, Loss: 0.2398, Test Acc: 0.91
Epoch 9/20, Loss: 0.2345, Test Acc: 0.91
Epoch 10/20, Loss: 0.2279, Test Acc: 0.92
Epoch 11/20, Loss: 0.2024, Test Acc: 0.92
Epoch 12/20, Loss: 0.1965, Test Acc: 0.93
Epoch 13/20, Loss: 0.1924, Test Acc: 0.93
Epoch 14/20, Loss: 0.1869, Test Acc: 0.92
Epoch 15/20, Loss: 0.1832, Test Acc: 0.93
Epoch 16/20, Loss: 0.1819, Test Acc: 0.93
Epoch 17/20, Loss: 0.1795, Test Acc: 0.93
Epoch 18/20, Loss: 0.1762, Test Acc: 0.92
Epoch 19/20, Loss: 0.1755, Test Acc: 0.92
Epoch 20/20, Loss: 0.1749, Test Acc: 0.93


In [17]:
# predict on test set and evaluate
tester_func(model, test_loader, device)

Accuracy: 0.9289

Classification Report:
              precision    recall  f1-score   support

 T-shirt/top       0.87      0.91      0.89      1000
     Trouser       1.00      0.99      0.99      1000
    Pullover       0.87      0.91      0.89      1000
       Dress       0.92      0.94      0.93      1000
        Coat       0.86      0.92      0.89      1000
      Sandal       0.98      0.99      0.99      1000
       Shirt       0.86      0.70      0.77      1000
     Sneaker       0.96      0.98      0.97      1000
         Bag       0.99      0.99      0.99      1000
  Ankle boot       0.98      0.96      0.97      1000

    accuracy                           0.93     10000
   macro avg       0.93      0.93      0.93     10000
weighted avg       0.93      0.93      0.93     10000


Confusion Matrix:
[[907   1  18  13   2   2  53   0   4   0]
 [  0 990   0   8   0   0   0   0   2   0]
 [ 15   0 912   7  41   0  25   0   0   0]
 [ 11   1   7 945  23   1  12   0   0   0]
 [  1   0

Ahem, well that fell on its face!

Note that we added 3 extra tricks in this last experiments, so any of the three could be the culprit! It's likely not the data augmentation that is to blame, which leaves us with learning rate scheduling and the weight decay. You figure out which!

-----
## 5. Fine-tune a Hugging Face Computer Vision Model (PyTorch)

In [18]:
def trainer_func_hf(model, image_processor, optimizer, train_loader, test_loader, device, epochs=10, scheduler=None):
    """
    Fine-tuning function for Hugging Face models
    """
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        
        for images, labels in train_loader:
            # Convert images from normalized [-1, 1] back to [0, 1] for preprocessing
            images = (images + 1) / 2.0
            
            # Convert to PIL Images and preprocess
            batch_images = []
            for img in images:
                # Convert to grayscale PIL Image
                img_np = (img.squeeze().cpu().numpy() * 255).astype(np.uint8)
                pil_img = Image.fromarray(img_np, mode='L').convert('RGB')  # Convert to RGB
                batch_images.append(pil_img)
            
            # Preprocess images using the image processor
            inputs = image_processor(images=batch_images, return_tensors="pt")
            pixel_values = inputs['pixel_values'].to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(pixel_values=pixel_values)
            logits = outputs.logits
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        
        # Evaluate on test set
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in test_loader:
                # Convert images from normalized [-1, 1] back to [0, 1]
                images = (images + 1) / 2.0
                
                # Convert to PIL Images
                batch_images = []
                for img in images:
                    img_np = (img.squeeze().cpu().numpy() * 255).astype(np.uint8)
                    pil_img = Image.fromarray(img_np, mode='L').convert('RGB')
                    batch_images.append(pil_img)
                
                # Preprocess and predict
                inputs = image_processor(images=batch_images, return_tensors="pt")
                pixel_values = inputs['pixel_values'].to(device)
                
                outputs = model(pixel_values=pixel_values)
                logits = outputs.logits
                _, predicted = torch.max(logits, 1)
                
                total += labels.size(0)
                correct += (predicted == labels.to(device)).sum().item()
        
        acc = correct / total
        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader):.4f}, Test Acc: {acc:.2f}")
        
        if scheduler:
            scheduler.step()

In [19]:
def tester_func_hf(model, image_processor, test_loader, device):
    """
    Testing function for Hugging Face models
    """
    model.eval()
    y_true, y_pred = [], []
    
    with torch.no_grad():
        for images, labels in test_loader:
            # Convert images from normalized [-1, 1] back to [0, 1]
            images = (images + 1) / 2.0
            
            # Convert to PIL Images
            batch_images = []
            for img in images:
                img_np = (img.squeeze().cpu().numpy() * 255).astype(np.uint8)
                pil_img = Image.fromarray(img_np, mode='L').convert('RGB')
                batch_images.append(pil_img)
            
            # Preprocess and predict
            inputs = image_processor(images=batch_images, return_tensors="pt")
            pixel_values = inputs['pixel_values'].to(device)
            
            outputs = model(pixel_values=pixel_values)
            logits = outputs.logits
            _, predicted = torch.max(logits, 1)
            
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(predicted.cpu().numpy())
    
    # Evaluate
    classes = [
        "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
        "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
    ]
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=classes))
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

In [ ]:
print("\n" + "=" * 80)
print("FINE-TUNING HUGGING FACE MODEL")
print("=" * 80)

# Load pre-trained model and processor
# Using ViT (Vision Transformer) as an example - you can change to other models like:
# - "google/mobilenet_v2_1.0_224"     -->  3.5 M parameters
# - "microsoft/resnet-18"             -->   11 M parameters
# - "google/vit-base-patch16-224"     -->   86 M parameters
# - "facebook/convnext-tiny-224"      -->   ??
# - "facebook/deit-tiny-patch16-224"  -->   ??

model_name = "google/mobilenet_v2_1.0_224"
print(f"\nLoading model: {model_name}")

# Load image processor and model
image_processor = AutoImageProcessor.from_pretrained(model_name)
# image_processor = AutoImageProcessor.from_pretrained(model_name, use_fast=True)  # google/vit-base-patch16-224 has fast mode

# Load model and modify the classifier head for 10 classes
hf_model = AutoModelForImageClassification.from_pretrained(
    model_name,
    num_labels=10,
    ignore_mismatched_sizes=True  # This allows changing the number of output classes
)
hf_model = hf_model.to(device)

# Set up optimizer with a lower learning rate for fine-tuning
hf_optimizer = optim.AdamW(hf_model.parameters(), lr=2e-5)

# Optional: Use a learning rate scheduler
hf_scheduler = optim.lr_scheduler.CosineAnnealingLR(hf_optimizer, T_max=10)

# Smaller batch sizes for Hugging Face models (better for RTX 3070 with 8GB VRAM)
train_loader_hf = torch.utils.data.DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader_hf = torch.utils.data.DataLoader(test_dataset, batch_size=500, shuffle=False)

# Fine-tune the model
print("\nFine-tuning starts...")
start_time = time.time()
trainer_func_hf(
    hf_model, 
    image_processor, 
    hf_optimizer, 
    train_loader_hf, 
    test_loader_hf, 
    device, 
    epochs=3,
    scheduler=hf_scheduler
)
end_time = time.time()
print(f"Fine-tuning completed in {(end_time - start_time)/60:.2f} minutes.")

# Evaluate the fine-tuned model
print("\nHugging Face Model Evaluation:")
tester_func_hf(hf_model, image_processor, test_loader, device)

# # Save the fine-tuned model (optional)
# print("\nSaving fine-tuned model...")
# hf_model.save_pretrained("./fashion_mnist_finetuned")
# image_processor.save_pretrained("./fashion_mnist_finetuned")
# print("Model saved to ./fashion_mnist_finetuned")


FINE-TUNING HUGGING FACE MODEL

Loading model: google/mobilenet_v2_1.0_224


Some weights of MobileNetV2ForImageClassification were not initialized from the model checkpoint at google/mobilenet_v2_1.0_224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1001]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.weight: found shape torch.Size([1001, 1280]) in the checkpoint and torch.Size([10, 1280]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Fine-tuning starts...
Epoch 1/3, Loss: 0.4802, Test Acc: 0.90
Epoch 2/3, Loss: 0.2483, Test Acc: 0.90
Epoch 3/3, Loss: 0.1923, Test Acc: 0.93
Fine-tuning completed in 22.24 minutes.

Hugging Face Model Evaluation:
Accuracy: 0.925

Classification Report:
              precision    recall  f1-score   support

 T-shirt/top       0.83      0.93      0.87      1000
     Trouser       1.00      0.97      0.99      1000
    Pullover       0.89      0.88      0.89      1000
       Dress       0.90      0.94      0.92      1000
        Coat       0.88      0.91      0.90      1000
      Sandal       0.99      0.98      0.99      1000
       Shirt       0.83      0.72      0.77      1000
     Sneaker       0.95      0.98      0.97      1000
         Bag       0.99      0.98      0.99      1000
  Ankle boot       0.99      0.95      0.97      1000

    accuracy                           0.93     10000
   macro avg       0.93      0.93      0.92     10000
weighted avg       0.93      0.93      0.

-----
## 6. Fine-tune a Hugging Face Computer Vision Model (Transformers)

In [ ]:
# from datasets import load_dataset
# from transformers import TrainingArguments, Trainer
# import evaluate
# import os

# model_name = "google/mobilenet_v2_1.0_224"
# # model_name = "facebook/deit-tiny-patch16-224"

# # load previously trained model if exists
# model_dir = "./deit-fmnist/checkpoint-7500/"  # where a saved model would exist is this cell block was run previously
# if os.path.isdir(model_dir):
#     model_name = model_dir
# print(f"Using model: {model_name}")

# # -------------------------------
# # 1. Load Fashion-MNIST from Hugging Face
# # -------------------------------
# dataset = load_dataset("fashion_mnist")

# # Classes for reference (10 labels)
# labels = dataset["train"].features["label"].names
# num_labels = len(labels)
# print("Labels:", labels)

# # -------------------------------
# # 2. Load Pretrained Lightweight Model
# # -------------------------------
# processor = AutoImageProcessor.from_pretrained(model_name)
# model = AutoModelForImageClassification.from_pretrained(
#     model_name,
#     num_labels=num_labels,
#     ignore_mismatched_sizes=True,
# )

# # -------------------------------
# # 3. Preprocessing Function
# # -------------------------------
# def preprocess_function(examples):
#     # Convert grayscale (1-channel) → RGB (3-channel) and resize to 224x224
#     images = [img.convert("RGB") for img in examples["image"]]
#     return processor(images, return_tensors="pt")

# # Apply preprocessing
# train_ds = dataset["train"].map(preprocess_function, batched=True)
# test_ds = dataset["test"].map(preprocess_function, batched=True)

# # Set dataset format for PyTorch
# train_ds.set_format(type="torch", columns=["pixel_values", "label"])
# test_ds.set_format(type="torch", columns=["pixel_values", "label"])

# # -------------------------------
# # 4. Metrics
# # -------------------------------
# metric = evaluate.load("accuracy")

# def compute_metrics(eval_pred):
#     logits, labels = eval_pred
#     preds = np.argmax(logits, axis=-1)
#     return metric.compute(predictions=preds, references=labels)

# # -------------------------------
# # 5. Training Configuration
# # -------------------------------
# training_args = TrainingArguments(
#     output_dir=model_dir,
#     evaluation_strategy="epoch",
#     save_strategy="epoch",
#     learning_rate=2e-5,
#     per_device_train_batch_size=8,  # fits comfortably in 8 GB VRAM
#     per_device_eval_batch_size=8,
#     num_train_epochs=3,
#     weight_decay=0.01,
#     load_best_model_at_end=True,
#     metric_for_best_model="accuracy",
#     logging_dir="./logs",
#     logging_steps=5000,
#     save_total_limit=2,                     # keep only best + latest checkpoint
#     fp16=True,                              # ✅ mixed precision → saves VRAM, faster on RTX GPUs
#     gradient_accumulation_steps=2,          # ✅ simulates batch size 16 effectively
#     warmup_ratio=0.1,                       # helps stabilize early training
#     report_to="none",                       # or "tensorboard" if you want logs
# )

# # -------------------------------
# # 6. Trainer API
# # -------------------------------
# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_ds,
#     eval_dataset=test_ds,
#     compute_metrics=compute_metrics,
# )

# # -------------------------------
# # 7. Fine-tune
# # -------------------------------
# if not os.path.isdir(model_dir):  # if NO previously trained model exists...
#     print("\nFine-tuning starts...")
#     start_time = time.time()
#     trainer.train()
#     end_time = time.time()
#     print(f"Fine-tuning completed in {(end_time - start_time)/60:.2f} minutes.")

# # -------------------------------
# # 8. Evaluate
# # -------------------------------
# results = trainer.evaluate()
# print("✅ Final Evaluation Results:", results)

-----
## What I was working with

In [23]:
print(torch.version.cuda)
print(torch.cuda.is_available())
device_id = torch.cuda.current_device()
print(device_id)
print(torch.cuda.get_device_name(device_id))

12.4
True
0
NVIDIA GeForce RTX 3070 Laptop GPU
